# Correctness Eval Results Analysis

Guardrail evaluation: how well do guardrails detect incorrect model predictions?
Supports W&B metric comparison + local pickle for detailed analysis (ROC/PR curves, etc.).

**Setup:** ensure you have wandb configured (`wandb login` or API key in `.env`).

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import pandas as pd
import wandb.errors

import pyine.evals.analysis_common
import pyine.evals.correctness
import pyine.evals.correctness.analysis
import pyine.evals.persistence

In [ ]:
WANDB_PROJECT = "pyine-tests"
RUN_FILTERS = {"state": "finished"}
TARGET_EVAL_SUBSET_NAME = "guardrail_test"
TARGET_FPR = 0.01
selected_run_idx = 0

# set if the run evaluates multiple guardrail types; None for single-type evals
GUARDRAIL_TYPE_NAME: str | None = None

# set to a local pickle path to load a CorrectnessEvalResult from disk
RESULT_PATH: str | None = None  # e.g. "logs/evals/guardrail_test.pkl"

In [ ]:
# load local pickle result if available
local_result = None
if RESULT_PATH is not None:
    local_result = pyine.evals.persistence.load_eval_result(
        pathlib.Path(RESULT_PATH),
        expected_type=pyine.evals.correctness.CorrectnessEvalResult,
    )
    print(f"Loaded local result with {len(local_result.aggregated.per_run)} run(s)")
    local_summary = pyine.evals.correctness.analysis.eval_result_to_summary(
        local_result,
        subset_name=TARGET_EVAL_SUBSET_NAME,
        source_path=pathlib.Path(RESULT_PATH) if RESULT_PATH else None,
        guardrail_type_name=GUARDRAIL_TYPE_NAME,
    )
    summaries = [local_summary]
    runs = []  # ensure runs/summaries are consistent
else:  # fetch runs from W&B (graceful degradation: continues with pickle if W&B unavailable)
    runs = []
    summaries = []
    try:
        runs = pyine.evals.analysis_common.fetch_runs(
            project=WANDB_PROJECT,
            filters=RUN_FILTERS if RUN_FILTERS else None,
            per_page=20,
        )
        print(f"Found {len(runs)} runs:")
    except (wandb.errors.CommError, wandb.errors.UsageError, ConnectionError, TimeoutError, OSError) as wandb_err:
        print(f"W&B fetch skipped ({type(wandb_err).__name__}: {wandb_err})")
        if local_result is None:
            raise  # no fallback available, re-raise
        runs = []
    # per-run summary parsing is outside the try block so data/logic errors propagate
    for run in runs:
        print(f"\t{run.group}/{run.name} ({run.url}) created at {run.created_at}")
        type_names = pyine.evals.correctness.analysis.detect_guardrail_type_names(run, TARGET_EVAL_SUBSET_NAME)
        type_to_use = GUARDRAIL_TYPE_NAME
        if type_names is not None and type_to_use is None and len(type_names) > 1:
            print(f"Skipping run {run.name}: multiple guardrail types {type_names}. Set GUARDRAIL_TYPE_NAME to select one.")
            continue
        if type_names is not None and type_to_use is None:
            type_to_use = type_names[0]
        if type_to_use is not None and type_names is not None and type_to_use not in type_names:
            print(f"Skipping run {run.name}: requested type {type_to_use!r} not in {type_names}")
            continue
        summaries.append(
            pyine.evals.correctness.analysis.fetch_correctness_eval_summary(
                run,
                TARGET_EVAL_SUBSET_NAME,
                type_name=type_to_use,
            )
        )

if summaries:
    df = pyine.evals.correctness.analysis.summarize_correctness_runs_to_dataframe(summaries)
else:
    df = pd.DataFrame()

df  # noqa: B018 (for display purposes)

In [ ]:
# threshold-free metrics comparison (W&B)
if summaries:
    fig = pyine.evals.correctness.analysis.plot_metric_comparison(summaries, "auroc")
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_metric_comparison(summaries, "average_precision")
    plt.tight_layout()
    plt.show()
else:
    print("no runs found to visualize")

In [ ]:
# ROC & PR curves (requires local result)
if local_result is not None:
    fig = pyine.evals.correctness.analysis.plot_roc_curves(local_result.aggregated.per_run)
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_pr_curves(local_result.aggregated.per_run)
    plt.tight_layout()
    plt.show()
else:
    print("ROC/PR curves require a local CorrectnessEvalResult.")
    print("Set RESULT_PATH to a pickle file produced by the eval pipeline.")

In [ ]:
# operating point analysis at target FPR
if summaries:
    fig = pyine.evals.correctness.analysis.plot_sample_level_metrics(summaries, TARGET_FPR)
    plt.tight_layout()
    plt.show()

# detailed confusion matrix from local result
if local_result is not None:
    # show metrics for run 0 as an example operating point
    first_run = local_result.aggregated.per_run[0]
    if TARGET_FPR in first_run.attempt_metrics and TARGET_FPR in first_run.sample_metrics:
        fig = pyine.evals.correctness.analysis.plot_operating_point_summary(
            first_run.attempt_metrics[TARGET_FPR],
            first_run.sample_metrics[TARGET_FPR],
            title=f"Operating Point (FPR={TARGET_FPR})",
        )
        plt.show()

In [ ]:
# category breakdown
if summaries:
    if not (0 <= selected_run_idx < len(summaries)):
        raise ValueError(f"selected_run_idx={selected_run_idx} is out of range for {len(summaries)} run(s)")
    selected_summary = summaries[selected_run_idx]

    fig = pyine.evals.correctness.analysis.plot_category_breakdown(selected_summary, "auroc")
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_category_breakdown(
        selected_summary,
        "tpr",
        target_fpr=TARGET_FPR,
    )
    plt.tight_layout()
    plt.show()

    fig = pyine.evals.correctness.analysis.plot_category_breakdown(
        selected_summary,
        "guarded_pass_rate",
        target_fpr=TARGET_FPR,
    )
    plt.tight_layout()
    plt.show()
else:
    print("no runs found to analyze")

In [ ]:
# difficulty-conditioned analysis (requires local result)
if local_result is not None and local_result.aggregated.difficulty_stats is not None:
    fig = pyine.evals.correctness.analysis.plot_difficulty_analysis(
        local_result.aggregated.difficulty_stats,
    )
    plt.tight_layout()
    plt.show()
else:
    print("No difficulty stats available (requires local result with difficulty data)")

In [ ]:
# verification cost analysis (requires local result)
if local_result is not None and local_result.aggregated.verification_cost_stats is not None:
    if TARGET_FPR in local_result.aggregated.verification_cost_stats:
        fig = pyine.evals.correctness.analysis.plot_cost_analysis(
            local_result.aggregated.verification_cost_stats[TARGET_FPR],
        )
        plt.tight_layout()
        plt.show()
    else:
        print(f"No cost data at FPR={TARGET_FPR}")
else:
    print("No verification cost stats available (requires local result with cost data)")

In [ ]:
# cross-run variability (W&B)
if summaries and len(summaries) > 1:
    fig = pyine.evals.correctness.analysis.plot_cross_run_variability(
        summaries,
        ["auroc", "average_precision", "tpr", "guarded_pass_rate", "unsafe_slip_rate"],
        target_fpr=TARGET_FPR,
    )
    plt.tight_layout()
    plt.show()
else:
    print("Need multiple runs for variability analysis")

In [ ]:
# one-by-one attempt browser (requires local pickle with per-attempt rows)
if local_result is None:
    print("Attempt browser requires RESULT_PATH (local CorrectnessEvalResult pickle)")
else:
    first_run = local_result.aggregated.per_run[0]
    record_lookup = local_result.aggregated.attempt_records_by_key or {}
    print("guardrail metadata for selected run:")
    print(first_run.guardrail_metadata)
    attempt_rows = [row.model_dump() for row in first_run.attempt_records]
    print(f"Loaded {len(attempt_rows)} attempt rows")
    if not attempt_rows:
        print("No attempt rows available in this result")
    else:
        attempts_df = pd.DataFrame(attempt_rows)
        try:
            import ipywidgets as widgets
            from IPython.display import display
        except ImportError:
            widgets = None
            display = print
        if widgets is None:
            print(attempts_df.head(1).T)
        else:
            sample_filter_widget = widgets.Text(value="", description="sample_id")
            label_filter_widget = widgets.Dropdown(
                options=["all", "correct", "incorrect"],
                value="all",
                description="label",
            )
            row_idx_widget = widgets.IntSlider(
                value=0,
                min=0,
                max=max(len(attempts_df) - 1, 0),
                step=1,
                description="row_idx",
                continuous_update=False,
            )
            output_widget = widgets.Output()

            def _render_attempt(sample_filter: str, label_filter: str, row_idx: int) -> None:
                filtered_df = attempts_df
                if sample_filter.strip():
                    filtered_df = filtered_df[
                        filtered_df["sample_id"].astype(str).str.contains(sample_filter, na=False)
                    ]
                if label_filter == "correct":
                    filtered_df = filtered_df[filtered_df["label"]]
                elif label_filter == "incorrect":
                    filtered_df = filtered_df[~filtered_df["label"]]
                with output_widget:
                    output_widget.clear_output(wait=True)
                    if filtered_df.empty:
                        print("No rows match current filters")
                        return
                    bounded_idx = min(row_idx, len(filtered_df) - 1)
                    selected_row = filtered_df.iloc[bounded_idx]
                    sample_id = str(selected_row["sample_id"])
                    attempt_index = int(selected_row["attempt_index"])
                    draw_index = selected_row.get("draw_index")
                    scored_attempt_key = None
                    if draw_index is not None and not pd.isna(draw_index):
                        scored_attempt_key = (sample_id, attempt_index, int(draw_index))
                    record_payload = record_lookup.get(scored_attempt_key) if scored_attempt_key is not None else None
                    if record_payload is None:
                        # compatibility fallback for older pickles keyed by (sample_id, attempt_index)
                        record_payload = record_lookup.get((sample_id, attempt_index))
                    print(f"filtered row {bounded_idx + 1}/{len(filtered_df)}")
                    display(selected_row.to_frame("value"))
                    if isinstance(record_payload, dict):
                        print("\n--- raw record payload ---")
                        display(pd.Series(record_payload).to_frame("value"))
                    if isinstance(selected_row.get("attempt_metadata"), dict):
                        print("\n--- guardrail attempt metadata ---")
                        display(pd.Series(selected_row["attempt_metadata"]).to_frame("value"))

            _attempt_browser_link = widgets.interactive_output(
                _render_attempt,
                {
                    "sample_filter": sample_filter_widget,
                    "label_filter": label_filter_widget,
                    "row_idx": row_idx_widget,
                },
            )
            display(sample_filter_widget)
            display(label_filter_widget)
            display(row_idx_widget)
            display(output_widget)